In [1]:
import pandas as pd
import numpy as np

# Load data

calls = pd.read_csv("../data/calls.csv")
sms = pd.read_csv("../data/sms.csv")

bt = pd.read_csv(
    "../data/bt_symmetric.csv",
    comment="#",
    header=None,
    names=["timestamp", "user_a", "user_b", "rssi"]
)

friends = pd.read_csv(
    "../data/fb_friends.csv",
    comment="#",
    header=None,
    names=["user_a", "user_b"]
)

genders = pd.read_csv(
    "../data/genders.csv",
    comment="#",
    header=None,
    names=["user", "female"]
)

In [2]:
# Create the call outcome:
# 1 = answered call
# 0 = missed call
calls["answered"] = (calls["duration"] != -1).astype(int)

# Sort temporal datasets chronologically
# This is important because all derived interaction features
# must only use information that occurred before each call.
calls = calls.sort_values("timestamp").reset_index(drop=True)
sms = sms.sort_values("timestamp").reset_index(drop=True)
bt = bt.sort_values("timestamp").reset_index(drop=True)

display(calls.head())

,timestamp,caller,callee,duration,answered
0,184,300,301,121,1
1,3920,512,299,670,1
2,5623,301,300,504,1
3,9252,401,457,-1,0
4,15466,512,0,5,1


In [3]:

# Each row represents one phone call.
# New features describing previous interactions will be added
# to this table throughout the preprocessing.

features = calls[
    ["timestamp", "caller", "callee", "answered"]
].copy()

print("Number of observations:", len(features))

display(features.head())


Number of observations: 3600


,timestamp,caller,callee,answered
0,184,300,301,1
1,3920,512,299,1
2,5623,301,300,1
3,9252,401,457,0
4,15466,512,0,1


Feature 1: Number of previous calls between participants

In [4]:
# For each call, count how many calls have previously occurred
# between the same two participants, regardless of direction.
#
# Only calls with an earlier timestamp are included.

previous_calls_between = []

# Dictionary that stores the number of previous calls for each pair
pair_call_counts = {}

# Process calls grouped by timestamp.
# This prevents calls occurring at exactly the same timestamp
# from being counted as previous interactions for each other.
for timestamp, group in calls.groupby("timestamp", sort=True):

    # First: calculate feature values using only earlier calls
    for idx, row in group.iterrows():

        # Create an unordered participant pair
        pair = tuple(sorted((row["caller"], row["callee"])))

        previous_calls_between.append(
            (idx, pair_call_counts.get(pair, 0))
        )

    # Second: after feature values are recorded,
    # update history with calls at the current timestamp
    for idx, row in group.iterrows():

        pair = tuple(sorted((row["caller"], row["callee"])))

        pair_call_counts[pair] = pair_call_counts.get(pair, 0) + 1


# Put values back in the correct row order
previous_calls_between = dict(previous_calls_between)

features["previous_calls_between"] = (
    features.index.map(previous_calls_between)
)

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between
0,184,300,301,1,0
1,3920,512,299,1,0
2,5623,301,300,1,1
3,9252,401,457,0,0
4,15466,512,0,1,0
5,15497,512,0,1,1
6,26400,19,47,1,0
7,31312,687,310,1,0
8,36265,300,301,1,2
9,37049,634,681,1,0


In [5]:
# Inspect the new feature

print(features["previous_calls_between"].describe())

print(
    "\nCalls with no previous calls between participants:",
    (features["previous_calls_between"] == 0).sum()
)

print(
    "Calls with at least one previous call between participants:",
    (features["previous_calls_between"] > 0).sum()
)

display(
    features.sort_values(
        "previous_calls_between",
        ascending=False
    ).head(10)
)

count    3600.000000
mean       11.190556
std        17.788774
min         0.000000
25%         1.000000
50%         4.000000
75%        13.000000
max       129.000000
Name: previous_calls_between, dtype: float64

Calls with no previous calls between participants: 621
Calls with at least one previous call between participants: 2979


,timestamp,caller,callee,answered,previous_calls_between
3539,2376133,578,289,1,129
3397,2305368,289,578,1,128
3395,2305107,578,289,1,127
3385,2304538,578,289,1,126
3325,2289288,578,289,1,125
3323,2288990,289,578,1,124
3322,2288828,578,289,1,123
3321,2288627,289,578,1,122
3319,2286558,289,578,1,121
3280,2231602,289,578,1,120


Feature 2: Previous answer rate between participants

In [6]:
# For each call, calculate the proportion of previous calls
# between the same two participants that were answered.
#
# Only calls with an earlier timestamp are included.

previous_answer_rate_between = []

# Store previous answered calls and total calls for each pair
pair_call_history = {}

for timestamp, group in calls.groupby("timestamp", sort=True):

    # First: calculate feature values from earlier calls only
    for idx, row in group.iterrows():

        pair = tuple(sorted((row["caller"], row["callee"])))

        if pair in pair_call_history:
            answered_count = pair_call_history[pair]["answered"]
            total_count = pair_call_history[pair]["total"]

            answer_rate = answered_count / total_count
        else:
            # No previous calls between the participants
            answer_rate = np.nan

        previous_answer_rate_between.append(
            (idx, answer_rate)
        )

    # Second: update history after all calls at this timestamp
    # have received their feature values
    for idx, row in group.iterrows():

        pair = tuple(sorted((row["caller"], row["callee"])))

        if pair not in pair_call_history:
            pair_call_history[pair] = {
                "answered": 0,
                "total": 0
            }

        pair_call_history[pair]["total"] += 1
        pair_call_history[pair]["answered"] += row["answered"]


# Add feature in correct row order
previous_answer_rate_between = dict(previous_answer_rate_between)

features["previous_answer_rate_between"] = (
    features.index.map(previous_answer_rate_between)
)

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between
0,184,300,301,1,0,NaN
1,3920,512,299,1,0,NaN
2,5623,301,300,1,1,1.0
3,9252,401,457,0,0,NaN
4,15466,512,0,1,0,NaN
5,15497,512,0,1,1,1.0
6,26400,19,47,1,0,NaN
7,31312,687,310,1,0,NaN
8,36265,300,301,1,2,1.0
9,37049,634,681,1,0,NaN


In [7]:
# Inspect the new feature

print(features["previous_answer_rate_between"].describe())

print(
    "\nCalls without previous call history:",
    features["previous_answer_rate_between"].isna().sum()
)

print(
    "Calls with previous call history:",
    features["previous_answer_rate_between"].notna().sum()
)

display(
    features[
        features["previous_calls_between"] > 0
    ][
        [
            "timestamp",
            "caller",
            "callee",
            "answered",
            "previous_calls_between",
            "previous_answer_rate_between"
        ]
    ].head(15)
)

count    2979.000000
mean        0.886047
std         0.216842
min         0.000000
25%         0.851852
50%         1.000000
75%         1.000000
max         1.000000
Name: previous_answer_rate_between, dtype: float64

Calls without previous call history: 621
Calls with previous call history: 2979


,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between
2,5623,301,300,1,1,1.0
5,15497,512,0,1,1,1.0
8,36265,300,301,1,2,1.0
13,38192,584,582,1,1,1.0
14,38277,466,183,1,1,1.0
15,38892,584,582,1,2,1.0
19,46716,269,267,1,1,1.0
22,48684,222,471,1,1,1.0
23,48810,222,471,1,2,1.0
25,49704,471,222,1,3,1.0


In [8]:
check = (
    (features["previous_calls_between"] == 0)
    ==
    features["previous_answer_rate_between"].isna()
)

print("Consistent:", check.all())

Consistent: True


Feature 3: Time since last call between perticipants 

In [9]:
# For each call, calculate how much time has passed since
# the most recent previous call between the same two participants.
#
# Only calls with an earlier timestamp are considered.

time_since_last_call = []

# Store the timestamp of the most recent call for each pair
last_call_time = {}

for timestamp, group in calls.groupby("timestamp", sort=True):

    # First: calculate feature values using earlier calls only
    for idx, row in group.iterrows():

        pair = tuple(sorted((row["caller"], row["callee"])))

        if pair in last_call_time:
            time_diff = timestamp - last_call_time[pair]
        else:
            time_diff = np.nan

        time_since_last_call.append(
            (idx, time_diff)
        )

    # Second: update history after all calls at this timestamp
    # have received their feature values
    for idx, row in group.iterrows():

        pair = tuple(sorted((row["caller"], row["callee"])))

        last_call_time[pair] = timestamp


# Add feature in correct row order
time_since_last_call = dict(time_since_last_call)

features["time_since_last_call"] = (
    features.index.map(time_since_last_call)
)

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call
0,184,300,301,1,0,NaN,NaN
1,3920,512,299,1,0,NaN,NaN
2,5623,301,300,1,1,1.0,5439.0
3,9252,401,457,0,0,NaN,NaN
4,15466,512,0,1,0,NaN,NaN
5,15497,512,0,1,1,1.0,31.0
6,26400,19,47,1,0,NaN,NaN
7,31312,687,310,1,0,NaN,NaN
8,36265,300,301,1,2,1.0,30642.0
9,37049,634,681,1,0,NaN,NaN


In [10]:
# Convert time since last call to more interpretable units

features["time_since_last_call_hours"] = (
    features["time_since_last_call"] / 3600
)

features["time_since_last_call_days"] = (
    features["time_since_last_call"] / (3600 * 24)
)

In [11]:
# Inspect the new feature

print(features["time_since_last_call_days"].describe())

print(
    "\nCalls without a previous call between participants:",
    features["time_since_last_call"].isna().sum()
)

print(
    "Calls with a previous call between participants:",
    features["time_since_last_call"].notna().sum()
)

display(
    features[
        features["previous_calls_between"] > 0
    ][
        [
            "timestamp",
            "caller",
            "callee",
            "previous_calls_between",
            "previous_answer_rate_between",
            "time_since_last_call_hours",
            "time_since_last_call_days"
        ]
    ].head(15)
)

count    2979.000000
mean        1.451021
std         3.204104
min         0.000023
25%         0.001997
50%         0.040903
75%         1.091557
max        26.169815
Name: time_since_last_call_days, dtype: float64

Calls without a previous call between participants: 621
Calls with a previous call between participants: 2979


,timestamp,caller,callee,previous_calls_between,previous_answer_rate_between,time_since_last_call_hours,time_since_last_call_days
2,5623,301,300,1,1.0,1.510833,0.062951
5,15497,512,0,1,1.0,0.008611,0.000359
8,36265,300,301,2,1.0,8.511667,0.354653
13,38192,584,582,1,1.0,0.011389,0.000475
14,38277,466,183,1,1.0,0.170556,0.007106
15,38892,584,582,2,1.0,0.194444,0.008102
19,46716,269,267,1,1.0,0.005000,0.000208
22,48684,222,471,1,1.0,0.063056,0.002627
23,48810,222,471,2,1.0,0.035000,0.001458
25,49704,471,222,3,1.0,0.248333,0.010347


In [12]:
check = (
    (features["previous_calls_between"] == 0)
    ==
    features["time_since_last_call"].isna()
)

print("Consistent:", check.all())

Consistent: True


Feauture 4: Receiver's overall previous answer rate

In [13]:
# For each call, calculate the receiver's answer rate
# based only on calls they received before the current call.
receiver_previous_call_count = []

receiver_previous_answer_rate = []

# Store previous incoming-call history for each receiver
receiver_history = {}

for timestamp, group in calls.groupby("timestamp", sort=True):

    # First: calculate feature values using earlier calls only
    for idx, row in group.iterrows():

        receiver = row["callee"]
        
        receiver_previous_call_count.append(
    (
        idx,
        receiver_history.get(receiver, {}).get("total", 0)
    )
)

        if receiver in receiver_history:
            answered_count = receiver_history[receiver]["answered"]
            total_count = receiver_history[receiver]["total"]

            answer_rate = answered_count / total_count
        else:
            answer_rate = np.nan

        receiver_previous_answer_rate.append(
            (idx, answer_rate)
        )

    # Second: update history after all calls at this timestamp
    # have received their feature values
    for idx, row in group.iterrows():

        receiver = row["callee"]

        if receiver not in receiver_history:
            receiver_history[receiver] = {
                "answered": 0,
                "total": 0
            }

        receiver_history[receiver]["total"] += 1
        receiver_history[receiver]["answered"] += row["answered"]


# Add feature in correct row order
receiver_previous_answer_rate = dict(receiver_previous_answer_rate)

features["receiver_previous_answer_rate"] = (
    features.index.map(receiver_previous_answer_rate)
)

display(features.head(10))

receiver_previous_call_count = dict(
    receiver_previous_call_count
)

features["receiver_previous_call_count"] = (
    features.index.map(receiver_previous_call_count)
)

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_answer_rate
0,184,300,301,1,0,NaN,NaN,NaN,NaN,NaN
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,NaN
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,NaN
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,NaN
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,NaN
5,15497,512,0,1,1,1.0,31.0,0.008611,0.000359,1.0
6,26400,19,47,1,0,NaN,NaN,NaN,NaN,NaN
7,31312,687,310,1,0,NaN,NaN,NaN,NaN,NaN
8,36265,300,301,1,2,1.0,30642.0,8.511667,0.354653,1.0
9,37049,634,681,1,0,NaN,NaN,NaN,NaN,NaN


In [14]:
# Inspect receiver previous answer rate

print(features["receiver_previous_answer_rate"].describe())

print(
    "\nCalls where receiver had no previous incoming calls:",
    features["receiver_previous_answer_rate"].isna().sum()
)

print(
    "Calls where receiver had previous incoming calls:",
    features["receiver_previous_answer_rate"].notna().sum()
)

display(
    features[
        features["receiver_previous_answer_rate"].notna()
    ][
        [
            "timestamp",
            "caller",
            "callee",
            "answered",
            "receiver_previous_answer_rate"
        ]
    ].head(15)
)

count    3120.000000
mean        0.900757
std         0.200582
min         0.000000
25%         0.900000
50%         1.000000
75%         1.000000
max         1.000000
Name: receiver_previous_answer_rate, dtype: float64

Calls where receiver had no previous incoming calls: 480
Calls where receiver had previous incoming calls: 3120


,timestamp,caller,callee,answered,receiver_previous_answer_rate
5,15497,512,0,1,1.0
8,36265,300,301,1,1.0
13,38192,584,582,1,1.0
14,38277,466,183,1,1.0
15,38892,584,582,1,1.0
22,48684,222,471,1,1.0
23,48810,222,471,1,1.0
27,50766,289,578,1,1.0
28,51682,289,578,1,1.0
30,55017,269,267,1,1.0


Feauture 5: Number of sms in the previous 48 hours

In [15]:
# For each phone call, count the number of SMS messages exchanged
# between the same two participants during the 48 hours before the call.
#
# SMS messages in both directions are included.
# Only messages sent before the current call are counted.

recent_sms_count_48h = []

# 48 hours in seconds
window_48h = 48 * 60 * 60

for idx, row in calls.iterrows():

    current_time = row["timestamp"]
    caller = row["caller"]
    callee = row["callee"]

    # Select SMS messages that:
    # 1. occurred before the phone call
    # 2. occurred within the previous 48 hours
    # 3. were exchanged between caller and callee in either direction

    sms_recent = sms[
        (sms["timestamp"] < current_time)
        & (sms["timestamp"] >= current_time - window_48h)
        & (
            (
                (sms["sender"] == caller)
                & (sms["recipient"] == callee)
            )
            |
            (
                (sms["sender"] == callee)
                & (sms["recipient"] == caller)
            )
        )
    ]

    recent_sms_count_48h.append(len(sms_recent))


# Add feature to feature table
features["recent_sms_count_48h"] = recent_sms_count_48h

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_answer_rate,receiver_previous_call_count,recent_sms_count_48h
0,184,300,301,1,0,NaN,NaN,NaN,NaN,NaN,0,0
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,NaN,0,7
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,NaN,0,1
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,NaN,0,0
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,NaN,0,28
5,15497,512,0,1,1,1.0,31.0,0.008611,0.000359,1.0,1,28
6,26400,19,47,1,0,NaN,NaN,NaN,NaN,NaN,0,4
7,31312,687,310,1,0,NaN,NaN,NaN,NaN,NaN,0,1
8,36265,300,301,1,2,1.0,30642.0,8.511667,0.354653,1.0,1,8
9,37049,634,681,1,0,NaN,NaN,NaN,NaN,NaN,0,0


In [16]:
# Inspect the new feature

print(features["recent_sms_count_48h"].describe())

print(
    "\nCalls with no SMS between participants in previous 48h:",
    (features["recent_sms_count_48h"] == 0).sum()
)

print(
    "Calls with at least one SMS between participants in previous 48h:",
    (features["recent_sms_count_48h"] > 0).sum()
)

display(
    features.sort_values(
        "recent_sms_count_48h",
        ascending=False
    )[
        [
            "timestamp",
            "caller",
            "callee",
            "answered",
            "recent_sms_count_48h"
        ]
    ].head(15)
)

count    3600.000000
mean        9.184444
std        21.139224
min         0.000000
25%         0.000000
50%         2.000000
75%         9.000000
max       326.000000
Name: recent_sms_count_48h, dtype: float64

Calls with no SMS between participants in previous 48h: 1484
Calls with at least one SMS between participants in previous 48h: 2116


,timestamp,caller,callee,answered,recent_sms_count_48h
1627,1112450,617,279,1,326
320,311559,279,617,1,324
321,311708,279,617,1,324
1644,1117072,279,617,1,300
3068,2116764,176,578,1,196
3028,2058285,176,578,1,191
3545,2381068,176,578,1,190
2679,1804944,617,279,1,189
3027,2057312,578,176,1,188
3026,2057181,578,176,1,187


Feature 6: Time since last sms between participants

In [17]:
# For each phone call, calculate the time since the most recent
# previous SMS exchanged between the same two participants.
#
# SMS messages in both directions are considered.
# Only SMS messages sent before the phone call are used.

time_since_last_sms = []

for idx, row in calls.iterrows():

    current_time = row["timestamp"]
    caller = row["caller"]
    callee = row["callee"]

    # Find all earlier SMS messages between the two participants
    previous_sms = sms[
        (sms["timestamp"] < current_time)
        & (
            (
                (sms["sender"] == caller)
                & (sms["recipient"] == callee)
            )
            |
            (
                (sms["sender"] == callee)
                & (sms["recipient"] == caller)
            )
        )
    ]

    if len(previous_sms) > 0:
        last_sms_time = previous_sms["timestamp"].max()
        time_diff = current_time - last_sms_time
    else:
        time_diff = np.nan

    time_since_last_sms.append(time_diff)


# Add feature to feature table
features["time_since_last_sms"] = time_since_last_sms

# Convert to more interpretable units
features["time_since_last_sms_hours"] = (
    features["time_since_last_sms"] / 3600
)

features["time_since_last_sms_days"] = (
    features["time_since_last_sms"] / (3600 * 24)
)

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_answer_rate,receiver_previous_call_count,recent_sms_count_48h,time_since_last_sms,time_since_last_sms_hours,time_since_last_sms_days
0,184,300,301,1,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,NaN,0,7,2.0,0.000556,0.000023
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,NaN,0,1,15.0,0.004167,0.000174
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,NaN,0,28,130.0,0.036111,0.001505
5,15497,512,0,1,1,1.0,31.0,0.008611,0.000359,1.0,1,28,161.0,0.044722,0.001863
6,26400,19,47,1,0,NaN,NaN,NaN,NaN,NaN,0,4,16350.0,4.541667,0.189236
7,31312,687,310,1,0,NaN,NaN,NaN,NaN,NaN,0,1,324.0,0.090000,0.003750
8,36265,300,301,1,2,1.0,30642.0,8.511667,0.354653,1.0,1,8,64.0,0.017778,0.000741
9,37049,634,681,1,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN


In [18]:
# Inspect the new feature

print(features["time_since_last_sms_days"].describe())

print(
    "\nCalls with no previous SMS between participants:",
    features["time_since_last_sms"].isna().sum()
)

print(
    "Calls with at least one previous SMS between participants:",
    features["time_since_last_sms"].notna().sum()
)

display(
    features[
        features["time_since_last_sms"].notna()
    ][
        [
            "timestamp",
            "caller",
            "callee",
            "answered",
            "recent_sms_count_48h",
            "time_since_last_sms_hours",
            "time_since_last_sms_days"
        ]
    ].head(15)
)

count    2857.000000
mean        1.992330
std         3.700466
min         0.000012
25%         0.014884
50%         0.226586
75%         2.093819
max        26.647060
Name: time_since_last_sms_days, dtype: float64

Calls with no previous SMS between participants: 743
Calls with at least one previous SMS between participants: 2857


,timestamp,caller,callee,answered,recent_sms_count_48h,time_since_last_sms_hours,time_since_last_sms_days
1,3920,512,299,1,7,0.000556,0.000023
2,5623,301,300,1,1,0.004167,0.000174
4,15466,512,0,1,28,0.036111,0.001505
5,15497,512,0,1,28,0.044722,0.001863
6,26400,19,47,1,4,4.541667,0.189236
7,31312,687,310,1,1,0.090000,0.003750
8,36265,300,301,1,8,0.017778,0.000741
11,37760,225,574,1,1,0.320278,0.013345
35,58643,139,546,1,4,0.008611,0.000359
36,58753,139,546,1,4,0.039167,0.001632


Feature 7: recent Bluetooth prowimity

In [19]:
# We operationalize recent Bluetooth proximity as the number
# of Bluetooth observations between the two participants
# during the 48 hours before each phone call.

# Create an unordered participant pair so that A-B and B-A
# are treated as the same relationship.

bt["pair"] = list(
    zip(
        np.minimum(bt["user_a"], bt["user_b"]),
        np.maximum(bt["user_a"], bt["user_b"])
    )
)

# Store sorted Bluetooth timestamps for each participant pair
bt_pair_times = (
    bt.groupby("pair")["timestamp"]
    .apply(lambda x: np.sort(x.to_numpy()))
    .to_dict()
)

print("Number of participant pairs in Bluetooth data:", len(bt_pair_times))

Number of participant pairs in Bluetooth data: 80886


In [20]:
# 48 hours in seconds
window_48h = 48 * 60 * 60

recent_bluetooth_count_48h = []

for idx, row in calls.iterrows():

    current_time = row["timestamp"]

    pair = tuple(
        sorted((row["caller"], row["callee"]))
    )

    # Check whether this pair has any Bluetooth observations
    if pair in bt_pair_times:

        times = bt_pair_times[pair]

        # Find observations within:
        # current_time - 48h <= timestamp < current_time

        start_index = np.searchsorted(
            times,
            current_time - window_48h,
            side="left"
        )

        end_index = np.searchsorted(
            times,
            current_time,
            side="left"
        )

        count = end_index - start_index

    else:
        count = 0

    recent_bluetooth_count_48h.append(count)


features["recent_bluetooth_count_48h"] = (
    recent_bluetooth_count_48h
)

display(features.head(10))

,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_answer_rate,receiver_previous_call_count,recent_sms_count_48h,time_since_last_sms,time_since_last_sms_hours,time_since_last_sms_days,recent_bluetooth_count_48h
0,184,300,301,1,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,NaN,0,7,2.0,0.000556,0.000023,0
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,NaN,0,1,15.0,0.004167,0.000174,0
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,NaN,0,28,130.0,0.036111,0.001505,0
5,15497,512,0,1,1,1.0,31.0,0.008611,0.000359,1.0,1,28,161.0,0.044722,0.001863,0
6,26400,19,47,1,0,NaN,NaN,NaN,NaN,NaN,0,4,16350.0,4.541667,0.189236,74
7,31312,687,310,1,0,NaN,NaN,NaN,NaN,NaN,0,1,324.0,0.090000,0.003750,0
8,36265,300,301,1,2,1.0,30642.0,8.511667,0.354653,1.0,1,8,64.0,0.017778,0.000741,0
9,37049,634,681,1,0,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0


In [21]:
# Inspect the Bluetooth feature

print(
    features["recent_bluetooth_count_48h"].describe()
)

print(
    "\nCalls with no Bluetooth observations between participants in previous 48h:",
    (features["recent_bluetooth_count_48h"] == 0).sum()
)

print(
    "Calls with at least one Bluetooth observation between participants in previous 48h:",
    (features["recent_bluetooth_count_48h"] > 0).sum()
)

display(
    features.sort_values(
        "recent_bluetooth_count_48h",
        ascending=False
    )[
        [
            "timestamp",
            "caller",
            "callee",
            "answered",
            "recent_bluetooth_count_48h"
        ]
    ].head(15)
)

count    3600.000000
mean       60.740000
std        99.540884
min         0.000000
25%         0.000000
50%        12.000000
75%        82.000000
max       570.000000
Name: recent_bluetooth_count_48h, dtype: float64

Calls with no Bluetooth observations between participants in previous 48h: 1567
Calls with at least one Bluetooth observation between participants in previous 48h: 2033


,timestamp,caller,callee,answered,recent_bluetooth_count_48h
1008,839390,321,322,0,570
949,814259,321,322,1,570
2341,1626481,322,321,1,527
381,343159,301,300,1,516
2268,1601360,322,321,1,509
2765,1870123,301,300,1,500
2763,1870084,301,300,1,500
3102,2129148,47,449,1,493
3119,2134415,47,449,1,491
455,404998,301,300,1,491


Feature 8, 9 and 10: Time-related features

In [22]:
# The released Copenhagen Networks Study data uses relative
# timestamps. The observation period starts on a Sunday.
#
# We therefore derive:
# - hour of day
# - day of week
# - weekend / weekday

seconds_per_day = 24 * 60 * 60

# Number of whole days since the beginning of the dataset
day_number = (features["timestamp"] // seconds_per_day).astype(int)

# Hour within the day
features["hour_of_day"] = (
    (features["timestamp"] % seconds_per_day) / 3600
)

# Day of week:
# 0 = Sunday
# 1 = Monday
# ...
# 6 = Saturday
features["day_of_week_num"] = day_number % 7

day_names = {
    0: "Sunday",
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday"
}

features["day_of_week"] = (
    features["day_of_week_num"].map(day_names)
)

# Weekend = Saturday or Sunday
features["weekend"] = (
    features["day_of_week_num"].isin([0, 6]).astype(int)
)

display(
    features[
        [
            "timestamp",
            "hour_of_day",
            "day_of_week",
            "weekend"
        ]
    ].head(15)
)

,timestamp,hour_of_day,day_of_week,weekend
0,184,0.051111,Sunday,1
1,3920,1.088889,Sunday,1
2,5623,1.561944,Sunday,1
3,9252,2.570000,Sunday,1
4,15466,4.296111,Sunday,1
5,15497,4.304722,Sunday,1
6,26400,7.333333,Sunday,1
7,31312,8.697778,Sunday,1
8,36265,10.073611,Sunday,1
9,37049,10.291389,Sunday,1


In [23]:
# Inspect time-related features

print("Hour of day:")
print(features["hour_of_day"].describe())

print("\nCalls by day of week:")
print(features["day_of_week"].value_counts())

print("\nCalls by weekday/weekend:")
print(
    features["weekend"]
    .value_counts()
    .rename(index={0: "Weekday", 1: "Weekend"})
)

Hour of day:
count    3600.000000
mean       14.972728
std         4.819655
min         0.004722
25%        12.067153
50%        15.519861
75%        18.193056
max        23.997222
Name: hour_of_day, dtype: float64

Calls by day of week:
day_of_week
Friday       839
Thursday     643
Wednesday    459
Saturday     446
Monday       419
Tuesday      408
Sunday       386
Name: count, dtype: int64

Calls by weekday/weekend:
weekend
Weekday    2768
Weekend     832
Name: count, dtype: int64


In [24]:
# Inspect time-related features

print("Hour of day:")
print(features["hour_of_day"].describe())

print("\nCalls by day of week:")
print(features["day_of_week"].value_counts())

print("\nCalls by weekday/weekend:")
print(
    features["weekend"]
    .value_counts()
    .rename(index={0: "Weekday", 1: "Weekend"})
)

Hour of day:
count    3600.000000
mean       14.972728
std         4.819655
min         0.004722
25%        12.067153
50%        15.519861
75%        18.193056
max        23.997222
Name: hour_of_day, dtype: float64

Calls by day of week:
day_of_week
Friday       839
Thursday     643
Wednesday    459
Saturday     446
Monday       419
Tuesday      408
Sunday       386
Name: count, dtype: int64

Calls by weekday/weekend:
weekend
Weekday    2768
Weekend     832
Name: count, dtype: int64


Final feature table:

Motivation for excluding FB friendship status in the main feature:

The Copenhagen Networks Study provides Facebook friendship information as a static snapshot taken at the end of the  observation period.

Since our analysis only uses information that was available before each phone call, using this snapshot could introduce future information for calls that occurred earlier in the observation period. Therefore, Facebook friendship is excluded from the main feature set to avoid potential temporal leakage.

In [25]:
selected_features = [
    "timestamp",
    "caller",
    "callee",
    "answered",

    "previous_calls_between",
    "previous_answer_rate_between",

    "time_since_last_call",
    "time_since_last_call_hours",
    "time_since_last_call_days",
    "receiver_previous_call_count",

    "receiver_previous_answer_rate",

    "recent_sms_count_48h",
    "time_since_last_sms",
    "time_since_last_sms_hours",
    "time_since_last_sms_days",

    "recent_bluetooth_count_48h",

    "hour_of_day",
    "day_of_week",
    "weekend"
]

features_final = features[selected_features].copy()

print("Shape:", features_final.shape)

display(features_final.head())

print("\nMissing values:")
display(features_final.isna().sum())

Shape: (3600, 19)


,timestamp,caller,callee,answered,previous_calls_between,previous_answer_rate_between,time_since_last_call,time_since_last_call_hours,time_since_last_call_days,receiver_previous_call_count,receiver_previous_answer_rate,recent_sms_count_48h,time_since_last_sms,time_since_last_sms_hours,time_since_last_sms_days,recent_bluetooth_count_48h,hour_of_day,day_of_week,weekend
0,184,300,301,1,0,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,0,0.051111,Sunday,1
1,3920,512,299,1,0,NaN,NaN,NaN,NaN,0,NaN,7,2.0,0.000556,0.000023,0,1.088889,Sunday,1
2,5623,301,300,1,1,1.0,5439.0,1.510833,0.062951,0,NaN,1,15.0,0.004167,0.000174,0,1.561944,Sunday,1
3,9252,401,457,0,0,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,0,2.570000,Sunday,1
4,15466,512,0,1,0,NaN,NaN,NaN,NaN,0,NaN,28,130.0,0.036111,0.001505,0,4.296111,Sunday,1



Missing values:


timestamp                          0
caller                             0
callee                             0
answered                           0
previous_calls_between             0
previous_answer_rate_between     621
time_since_last_call             621
time_since_last_call_hours       621
time_since_last_call_days        621
receiver_previous_call_count       0
receiver_previous_answer_rate    480
recent_sms_count_48h               0
time_since_last_sms              743
time_since_last_sms_hours        743
time_since_last_sms_days         743
recent_bluetooth_count_48h         0
hour_of_day                        0
day_of_week                        0
weekend                            0
dtype: int64

In [26]:
print("Number of calls:", len(features_final))

print(
    "Previous calls range:",
    features_final["previous_calls_between"].min(),
    "-",
    features_final["previous_calls_between"].max()
)

print(
    "Previous answer rate range:",
    features_final["previous_answer_rate_between"].min(),
    "-",
    features_final["previous_answer_rate_between"].max()
)

print(
    "Receiver answer rate range:",
    features_final["receiver_previous_answer_rate"].min(),
    "-",
    features_final["receiver_previous_answer_rate"].max()
)

print(
    "Recent SMS count range:",
    features_final["recent_sms_count_48h"].min(),
    "-",
    features_final["recent_sms_count_48h"].max()
)

print(
    "Recent Bluetooth count range:",
    features_final["recent_bluetooth_count_48h"].min(),
    "-",
    features_final["recent_bluetooth_count_48h"].max()
)

print(
    "Hour of day range:",
    features_final["hour_of_day"].min(),
    "-",
    features_final["hour_of_day"].max()
)

Number of calls: 3600
Previous calls range: 0 - 129
Previous answer rate range: 0.0 - 1.0
Receiver answer rate range: 0.0 - 1.0
Recent SMS count range: 0 - 326
Recent Bluetooth count range: 0 - 570
Hour of day range: 0.004722222222222222 - 23.997222222222224


In [27]:
# Save processed feature table

features_final.to_csv(
    "../data/processed_features.csv",
    index=False
)

print("Processed feature table saved.")

Processed feature table saved.
